# Trabalho de DL - Tradutor (Transformer) - Versão usando Pytorch - G13

### 1. Importando as bibliotecas

In [1]:
import random

from datasets import load_dataset

import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F

import math

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

### 2. Escolhendo o Device

In [2]:
# Verificar se temos GPU disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Usando: cuda
GPU: Tesla T4


### 3. Hyperparâmetros

In [3]:
# Hyperparametros do trabalho
NUM_SAMPLES = 10000      # Número de amostras a serem usadas para treinamento do tokenizador
BATCH_SIZE = 32         # Tamanho de cada mini batch
VOCAB_SIZE = 1600       # Tamanho do vocabulário do Tokenizador
D_MODEL = 128           # Dimensões do modelo
NUM_HEADS = 4           # Número de cabeças do mecanismo de atenção
NUM_ENCODER_LAYERS = 2  # Número de camadas de encoder e decoder
D_FF = 512              # 2048
DROPOUT = 0.1           # Taxa de dropout

### 4. Carregando os dados

In [4]:
dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-pt"
)

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-pt/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  220kB            

en-pt/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-pt/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 87.2MB            

en-pt/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-pt/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  217kB            

en-pt/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})


In [6]:
# Um milhão de amostras é muita coisa, reduzindo o treino para 50.000


#train = dataset["train"].select(range(NUM_SAMPLES)) # As 50.000 primeiras

train = dataset["train"].shuffle(seed=13).select(range(NUM_SAMPLES)) # Escolha aleatória

valid = dataset["validation"]

test = dataset["test"]

In [ ]:
print(train[0]) # primeira (1)
print(train[-1]) # última (50.000)


{'translation': {'en': 'No, it was just Google.', 'pt': 'Foi só o Google.'}}
{'translation': {'en': 'O.Z., half of this is for you.', 'pt': 'O.Z., metade disto é para si.'}}


In [8]:
# Escolha 5 amostras da base de treino
for i in random.sample(range(NUM_SAMPLES), 5):

    exemplo = train[i]["translation"]

    print("Português :", exemplo["pt"])
    print("Inglês    :", exemplo["en"])
    print("-"*60)

Português : Ele não parece muito inteligente.
Inglês    : He doesn't sound that clever.
------------------------------------------------------------
Português : Leal, paciente.
Inglês    : Loyal, patient.
------------------------------------------------------------
Português : Posso perguntar uma coisa?
Inglês    : Can I ask you something?
------------------------------------------------------------
Português : Vês?
Inglês    : See? Uh, Lilo...
------------------------------------------------------------
Português : Não necessariamente do homicídio.
Inglês    : It wasn't necessarily from the murder.
------------------------------------------------------------


In [9]:
# Separando os idiomas
train_pt = [
    exemplo["translation"]["pt"]
    for exemplo in train
]

train_en = [
    exemplo["translation"]["en"]
    for exemplo in train
]

In [10]:
valid_pt = [
    exemplo["translation"]["pt"]
    for exemplo in valid
]

valid_en = [
    exemplo["translation"]["en"]
    for exemplo in valid
]

test_pt = [
    exemplo["translation"]["pt"]
    for exemplo in test
]

test_en = [
    exemplo["translation"]["en"]
    for exemplo in test
]

In [ ]:
print(train_pt[0])
print(train_en[0])

Foi só o Google.
No, it was just Google.


In [12]:
print(f"Treino    : {len(train_pt)}")
print(f"Validação : {len(valid_pt)}")
print(f"Teste     : {len(test_pt)}")

Treino    : 10000
Validação : 2000
Teste     : 2000


### 5. Tokenização

In [13]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [14]:
# Cria o corpus a partir das frases em Português e Inglês
corpus = train_pt + train_en


In [15]:
# Cria o Tokenizador
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=[
        "[PAD]",
        "[BOS]",
        "[EOS]",
        "[UNK]"
    ]
)

O tokenizador foi treinado utilizando um corpus bilíngue composto pelas sentenças dos conjuntos de treino em português e inglês. As sentenças foram fornecidas diretamente ao algoritmo BPE, que aprendeu um vocabulário de subpalavras a partir das frequências observadas no corpus.

In [16]:
# Treina o tokenizador
tokenizer.train_from_iterator(
    corpus,
    trainer=trainer
)

tokenizer.save("tokenizer.json")

In [17]:
tokenizer.get_vocab_size()

1600

In [18]:
# Verificação

#frase = "O transformador aprende padrões"
#frase = "I like to study"
frase = random.sample(corpus,1)[0]
print(frase)

encoding = tokenizer.encode(frase)

print("IDs:", encoding.ids)
print("Tokens:", encoding.tokens)

Yes, she is.
IDs: [1353, 15, 767, 225, 17]
Tokens: ['Yes', ',', 'she', 'is', '.']


In [19]:
# Decodificação
ids = encoding.ids

texto = tokenizer.decode(ids)

print(texto)

Yes , she is .


In [20]:
# teste de lote
frases = [
    "bom dia",
    "eu gosto de programação",
    "o transformer utiliza atenção",
    "como você está"
]

for frase in frases:
    ids = tokenizer.encode(frase).ids

    reconstruida = tokenizer.decode(ids)

    print("-" * 50)
    print("Original     :", frase)
    print("Reconstruída :", reconstruida)

--------------------------------------------------
Original     : bom dia
Reconstruída : bom dia
--------------------------------------------------
Original     : eu gosto de programação
Reconstruída : eu gos to de program ação
--------------------------------------------------
Original     : o transformer utiliza atenção
Reconstruída : o trans for mer utiliz a at en ção
--------------------------------------------------
Original     : como você está
Reconstruída : como você está


In [ ]:
# Inclusão de tokens para delimitar o início e fim da sentença
bos_id = tokenizer.token_to_id("[BOS]")
eos_id = tokenizer.token_to_id("[EOS]")

frase = "eu gosto de estudar"

ids = tokenizer.encode(frase).ids

encoder_input = ids

decoder_input = [bos_id] + ids

target = ids + [eos_id]

print("Encoder:", encoder_input)
print("Decoder:", decoder_input)
print("Target :", target)

Encoder: [416, 729, 230, 224, 281, 88, 742]
Decoder: [1, 416, 729, 230, 224, 281, 88, 742]
Target : [416, 729, 230, 224, 281, 88, 742, 2]


In [ ]:
frase = "O transformador é eficiente"

enc = tokenizer.encode(frase)

print("Frase :", frase)
print("Tokens:", enc.tokens)
print("IDs   :", enc.ids)
print("Decode:", tokenizer.decode(enc.ids))

Frase : O transformador é eficiente
Tokens: ['O', 'trans', 'forma', 'dor', 'é', 'e', 'fic', 'i', 'ente']
IDs   : [50, 670, 714, 1372, 134, 72, 358, 76, 298]
Decode: O trans forma dor é e fic i ente


### 6. Dataset e Dataloader

In [23]:
class TranslationDataset(Dataset):

    def __init__(
        self,
        samples, # Lista de frases em português e inglês
        tokenizer
    ):
        self.samples = samples
        self.src_samples, self.tgt_samples = zip(*samples)
        self.tokenizer = tokenizer
        self.bos_id = tokenizer.token_to_id("[BOS]")
        self.eos_id = tokenizer.token_to_id("[EOS]")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        src_sample = self.src_samples[idx]
        tgt_sample = self.tgt_samples[idx]

        src_ids = (
            [self.bos_id]
            + self.tokenizer.encode(src_sample).ids
            + [self.eos_id]
        )

        tgt_ids = (
            [self.bos_id]
            + self.tokenizer.encode(tgt_sample).ids
            + [self.eos_id]
        )

        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_ids, dtype=torch.long)
        )

In [24]:
# Função que inclui o padding nas amostras
from torch.nn.utils.rnn import pad_sequence

#PAD_ID = 0
PAD_ID = tokenizer.token_to_id("[PAD]")

def collate_fn(batch):

    src_batch, tgt_batch = zip(*batch)

    src_batch = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=PAD_ID
    )

    tgt_batch = pad_sequence(
        tgt_batch,
        batch_first=True,
        padding_value=PAD_ID
    )

    return src_batch, tgt_batch

In [25]:
# Cria o Dataset e o Dataloader de treinamento

from torch.utils.data import DataLoader
train_pairs = list(zip(train_pt, train_en))

train_dataset = TranslationDataset(train_pairs, tokenizer) # cria o dataset de treinamento

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
) # cria o dataloader de treinamento

In [26]:
len(train_dataset)

10000

In [ ]:
len(train_loader)

313

In [28]:
one_batch_src, one_batch_tgt = next(iter(train_loader))

print(one_batch_src[0])
print(one_batch_tgt[0])

tensor([   1,   38,  503, 1439,  134,  250, 1439,   15,  259,  635,   80,  293,
         138,   68,  224,  224, 1325,   91, 1318,   17,    2,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0])
tensor([   1,   40,  707, 1521,  267,  221,   86,  735, 1562,  414,  254,  291,
        1325,   91,   76,  810,   17,    2,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,

In [ ]:
print(tokenizer.decode(one_batch_src[0].tolist()))
print(100*'-')
print(tokenizer.decode(one_batch_tgt[0].tolist()))

C ada caso é um caso , com sua m ir í a de de comple x idades .
----------------------------------------------------------------------------------------------------
E ach case be ar s its own my ri ad comple x i ties .


In [ ]:
# Dataload de validação
valid_pairs = list(zip(valid_pt, valid_en))

valid_dataset = TranslationDataset(
    valid_pairs,
    tokenizer
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


In [ ]:
train_dataset[0]

(tensor([   1, 1222,   76,  755,   82,  915,   82,   74,  248,   17,    2]),
 tensor([  1, 368,  15, 244, 377, 494, 915,  82,  74, 248,  17,   2]))

In [32]:
src_lengths = [len(item[0]) for item in train_dataset] # obtem os tamanhos das amostras src
tgt_lengths = [len(item[1]) for item in train_dataset] # obtem os tamanhos das amostras tgt

print(max(src_lengths)) # obtem o tamanho da maior amostra src
print(max(tgt_lengths)) # obtem o tamanho da maior amostra tgt

print(sum(src_lengths) / len(src_lengths)) # calcula o tamanho médio das amostras src
print(sum(tgt_lengths) / len(tgt_lengths)) # calcula o tamanho médio das amostras tgt

628
527
21.8507
21.1719


In [ ]:
src, tgt = next(iter(train_loader))

print(src.shape)  # (32,xx)
print(tgt.shape)  # (32,yy)


torch.Size([32, 93])
torch.Size([32, 93])


In [ ]:
src[0].shape

torch.Size([93])

In [35]:
src[0]

tensor([   1, 1058,   39, 1068,   15,  265,  254,  748,   16,  241,  302,   68,
         295,  314,  224,  224,  393,  287,   22,   17,    2,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0])

In [36]:
print(tokenizer.decode(src[0].tolist()))
print(tokenizer.decode(tgt[0].tolist()))


Senhor D ata , di ri ja - se para a ce la de de ten ção 3 .
Mr . D ata , report to de ten tion ce ll th ree .


### 7. Embedding

In [37]:
import torch
import torch.nn as nn



embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)

# Testando
x = torch.randint(0, VOCAB_SIZE, (32, 20))

emb = embedding(x)

print(emb.shape)


torch.Size([32, 20, 128])


### 8. Positional Encoder

In [ ]:
# =====================================================
# Positional Encoding
# =====================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0, max_len, dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0, d_model, 2,
                dtype=torch.float
            ) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


In [39]:
pos = PositionalEncoding(128)

x = pos(emb)

### 9. Scaled Dot Product Attention

In [ ]:
# =====================================================
# Scaled Dot Product Attention
# =====================================================

class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, q, k, v, mask=None):

        d_k = q.size(-1)

        scores = torch.matmul(
            q, k.transpose(-2, -1)
        ) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, v)

        #return output, attention
        return output

### 10. Multi-Head Attention

In [ ]:
# =====================================================
# Multi Head Attention
# =====================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.w_o = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

    def split_heads(self, x):

        batch_size = x.size(0)

        x = x.view(
            batch_size,
            -1,
            self.num_heads,
            self.d_k
        )

        return x.transpose(1, 2)

    def combine_heads(self, x):

        batch_size = x.size(0)

        x = x.transpose(1, 2).contiguous()

        return x.view(
            batch_size,
            -1,
            self.d_model
        )

    def forward(self, q, k, v, mask=None):

        q = self.split_heads(self.w_q(q))
        k = self.split_heads(self.w_k(k))
        v = self.split_heads(self.w_v(v))

        #output, attn = self.attention(
        output = self.attention(
            q,
            k,
            v,
            mask
        )

        output = self.combine_heads(output)

        output = self.w_o(output)

        return output


### 11. Feed Forward Network

In [ ]:
# =====================================================
# Feed Forward Network
# =====================================================

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### 12. Encoder Layer

In [ ]:
# =====================================================
# Encoder Layer
# =====================================================

class EncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = PositionwiseFeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):

        attn_output = self.self_attn(
            x, x, x, src_mask
        )

        x = self.norm1(
            x + self.dropout(attn_output)
        )

        ff_output = self.ffn(x)

        x = self.norm2(
            x + self.dropout(ff_output)
        )

        return x

### 13. Decoder Layer

In [ ]:
# =====================================================
# Decoder Layer
# =====================================================

class DecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.cross_attn = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = PositionwiseFeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x,
        encoder_output,
        src_mask,
        tgt_mask
    ):

        attn = self.self_attn(
            x, x, x, tgt_mask
        )

        x = self.norm1(
            x + self.dropout(attn)
        )

        attn = self.cross_attn(
            x,
            encoder_output,
            encoder_output,
            src_mask
        )

        x = self.norm2(
            x + self.dropout(attn)
        )

        ff = self.ffn(x)

        x = self.norm3(
            x + self.dropout(ff)
        )

        return x


### 14. Encoder

In [ ]:
# =====================================================
# Encoder
# =====================================================

class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = PositionalEncoding(
            d_model
        )

        self.layers = nn.ModuleList(
            [
                EncoderLayer(
                    d_model,
                    num_heads,
                    d_ff,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, src, src_mask):

        x = self.embedding(src) * math.sqrt(
            self.d_model
        )

        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, src_mask)

        return x


### 15. Decoder

In [ ]:
# =====================================================
# Decoder
# =====================================================

class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = PositionalEncoding(
            d_model
        )

        self.layers = nn.ModuleList(
            [
                DecoderLayer(
                    d_model,
                    num_heads,
                    d_ff,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(
        self,
        tgt,
        encoder_output,
        src_mask,
        tgt_mask
    ):

        x = self.embedding(tgt) * math.sqrt(
            self.d_model
        )

        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(
                x,
                encoder_output,
                src_mask,
                tgt_mask
            )

        return x

### 16. Transformer

In [ ]:
# =====================================================
# Transformer Completo
# =====================================================

class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        num_layers=6,
        num_heads=8,
        d_ff=2048,
        dropout=0.1
    ):
        super().__init__()

        self.encoder = Encoder(
            src_vocab_size,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            dropout
        )

        self.decoder = Decoder(
            tgt_vocab_size,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            dropout
        )

        self.fc_out = nn.Linear(
            d_model,
            tgt_vocab_size
        )

    def forward(
        self,
        src,
        tgt,
        src_mask,
        tgt_mask
    ):

        encoder_output = self.encoder(
            src,
            src_mask
        )

        decoder_output = self.decoder(
            tgt,
            encoder_output,
            src_mask,
            tgt_mask
        )

        output = self.fc_out(
            decoder_output
        )

        return output


### 17. Máscaras

In [ ]:
# =====================================================
# Máscaras
# =====================================================

def create_padding_mask(seq, pad_idx=0):
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)


def create_causal_mask(size):

    mask = torch.tril(
        torch.ones(size, size)
    )

    return mask.bool().unsqueeze(0).unsqueeze(1)

### 18. Aplicação

In [49]:
src_vocab_size = VOCAB_SIZE
tgt_vocab_size = VOCAB_SIZE

model = Transformer(
    src_vocab_size,
    tgt_vocab_size
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)



In [ ]:
PAD_IDX = tokenizer.token_to_id("[PAD]")

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


In [51]:
src = torch.randint(
    1,
    src_vocab_size,
    (4, 20)
).to(device)

tgt = torch.randint(
    1,
    tgt_vocab_size,
    (4, 15)
).to(device)

src_mask = create_padding_mask(src).to(device)

tgt_input = tgt[:, :-1].to(device)

tgt_output = tgt[:, 1:].to(device)

tgt_mask = (
    create_padding_mask(tgt_input).to(device)
    & create_causal_mask(tgt_input.size(1)).to(device)
)

output = model(
    src,
    tgt_input,
    src_mask,
    tgt_mask
)

loss = criterion(
    output.reshape(-1, output.size(-1)),
    tgt_output.reshape(-1)
)
print(output.shape)
# (4, 14, VOCAB_SIZE)

torch.Size([4, 14, 1600])


### 19. Loop de treinamento

In [52]:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Transformer(
    src_vocab_size,
    tgt_vocab_size,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_ENCODER_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)

In [53]:
import time

def train_epoch(model, dataloader, optimizer, criterion, device, epoch):
    start = time.time()
    model.train()

    total_loss = 0.0
  
   
    for (batch_num, (src, tgt)) in enumerate(train_loader):

        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        src_mask = create_padding_mask(src).to(device)

        tgt_mask = (
            create_padding_mask(tgt_input)
            &
            create_causal_mask(tgt_input.size(1)).to(device)
        )

        optimizer.zero_grad()

        output = model(
            src,
            tgt_input,
            src_mask,
            tgt_mask
        )

        loss = criterion(
            output.reshape(-1, output.size(-1)),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()
        if batch_num % 50 == 0:
              print(f'Epoch {epoch + 1:02d} Batch {batch_num} ')
        
              
    print(f'Time taken for 1 epoch: {time.time() - start:.2f} secs')
    return total_loss / len(dataloader)

In [54]:
NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):

    loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
        epoch
    )

    print(f"Epoch {epoch+1:02d} | Loss = {loss:.4f}\n")

Epoch 01 Batch 0 
Epoch 01 Batch 50 
Epoch 01 Batch 100 
Epoch 01 Batch 150 
Epoch 01 Batch 200 
Epoch 01 Batch 250 
Epoch 01 Batch 300 
Time taken for 1 epoch: 8.14 secs
Epoch 01 | Loss = 6.1496

Epoch 02 Batch 0 
Epoch 02 Batch 50 
Epoch 02 Batch 100 
Epoch 02 Batch 150 
Epoch 02 Batch 200 
Epoch 02 Batch 250 
Epoch 02 Batch 300 
Time taken for 1 epoch: 8.24 secs
Epoch 02 | Loss = 5.5300

Epoch 03 Batch 0 
Epoch 03 Batch 50 
Epoch 03 Batch 100 
Epoch 03 Batch 150 
Epoch 03 Batch 200 
Epoch 03 Batch 250 
Epoch 03 Batch 300 
Time taken for 1 epoch: 8.10 secs
Epoch 03 | Loss = 5.3540

Epoch 04 Batch 0 
Epoch 04 Batch 50 
Epoch 04 Batch 100 
Epoch 04 Batch 150 
Epoch 04 Batch 200 
Epoch 04 Batch 250 
Epoch 04 Batch 300 
Time taken for 1 epoch: 7.59 secs
Epoch 04 | Loss = 5.2075

Epoch 05 Batch 0 
Epoch 05 Batch 50 
Epoch 05 Batch 100 
Epoch 05 Batch 150 
Epoch 05 Batch 200 
Epoch 05 Batch 250 
Epoch 05 Batch 300 
Time taken for 1 epoch: 7.96 secs
Epoch 05 | Loss = 5.0903



### 20. Salvamento do Modelo

In [55]:
torch.save(
    model.state_dict(),
    "translator.pt"
)

In [56]:
# Load
# model.load_state_dict(
#     torch.load("translator.pt")
# )

### 21. Inferência

In [57]:
# #MAX_LEN=128

# @torch.no_grad()
# def translate(sentence):

#     model.eval()

#     src = torch.tensor(
#         tokenizer.encode(sentence).ids,
#         dtype=torch.long
#     ).unsqueeze(0).to(device)

#     src_mask = create_padding_mask(src).to(device)

#     encoder_output = model.encoder(src, src_mask)

#     tgt = torch.tensor(
#         [[bos_id]],
#         device=device
#     )
#     max_len = min(len(src)+ 30, 128)
    
#     for _ in range(max_len):

#         tgt_mask = (
#             create_padding_mask(tgt)
#             &
#             create_causal_mask(tgt.size(1)).to(device)
#         )

#         decoder_output = model.decoder(
#             tgt,
#             encoder_output,
#             src_mask,
#             tgt_mask
#         )

#         logits = model.fc_out(decoder_output)

#         next_token = logits[:, -1].argmax(-1)

#         tgt = torch.cat(
#             [tgt, next_token.unsqueeze(1)],
#             dim=1
#         )

#         if next_token.item() == eos_id:
#             break
#     tokens = tgt[0].tolist()
    
#     tokens = [
#         t for t in tokens
#         if t not in (bos_id. eos_id, PAD_ID)
#     ]
#     return tokenizer.decode(tokens)

In [58]:
@torch.no_grad()
def translate(sentence, max_len=None):

    model.eval()

    src_ids = tokenizer.encode(sentence).ids

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    src_mask = create_padding_mask(src)

    encoder_output = model.encoder(src, src_mask)

    tgt = torch.tensor(
        [[bos_id]],
        dtype=torch.long,
        device=device
    )

    if max_len is None:
        max_len = min(len(src_ids) + 30, 128)

    for _ in range(max_len):

        tgt_mask = (
            create_padding_mask(tgt)
            &
            create_causal_mask(tgt.size(1)).to(device)
        )

        decoder_output = model.decoder(
            tgt,
            encoder_output,
            src_mask,
            tgt_mask
        )

        logits = model.fc_out(decoder_output)

        next_token = logits[:, -1].argmax(dim=-1)

        tgt = torch.cat(
            (tgt, next_token.unsqueeze(1)),
            dim=1
        )

        if next_token.item() == eos_id:
            break

    tokens = [
        t
        for t in tgt[0].tolist()
        if t not in (bos_id, eos_id, PAD_ID)
    ]

    return tokenizer.decode(tokens)

In [59]:
sample = train_dataset[0]

print(sample[0])


print(tokenizer.decode(sample[0].tolist()))
print(tokenizer.decode(sample[1].tolist()))

tensor([   1, 1222,   76,  755,   82,  915,   82,   74,  248,   17,    2])
Fo i só o Go o g le .
No , it was just Go o g le .


In [60]:
sentence = "Eu gosto de estudar"

translate(sentence)

"I ' t the be to the Commission the Commission the Commission the st of the Commission the Commission the st of the Commission the st of the st of the Commission the st of the Commission"

In [61]:
src = torch.tensor(
        tokenizer.encode(sentence).ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

In [62]:
src_mask = create_padding_mask(src)
encoder_output = model.encoder(src, src_mask)
encoder_output

tensor([[[-1.2558e+00, -1.3296e-01, -5.9145e-01, -1.0536e+00, -4.2571e-01,
           6.8191e-01, -2.0899e-01, -2.3110e+00, -7.3261e-01, -1.7305e+00,
           8.0636e-02, -4.4751e-01, -6.7321e-03, -1.9793e-01, -2.8541e-01,
          -3.7421e-01,  9.0150e-01, -5.6801e-01, -1.4851e-01,  8.6659e-02,
          -2.4552e-01, -1.2695e+00,  2.6564e-01, -4.8961e-01,  6.9509e-01,
          -1.9378e+00, -8.7738e-01,  9.3470e-01, -1.1137e+00,  1.3260e+00,
           9.7348e-01,  4.4493e-01, -2.0922e+00, -7.3128e-01, -8.5550e-02,
           1.7441e-02, -5.6026e-01, -2.4918e+00, -7.6218e-01, -2.6555e-01,
          -1.1588e-01, -1.7728e+00, -1.7023e+00,  3.7415e-01,  4.0995e-01,
          -1.5121e-01,  1.7932e+00,  1.5359e-01, -4.2861e-01,  7.9398e-01,
           8.7782e-01, -4.2605e-01,  2.9898e-01, -3.5014e-01, -1.7476e-01,
           3.5509e-01, -1.8337e-01,  8.8426e-02, -3.0289e-01,  1.4917e+00,
           1.5561e+00, -1.1140e+00,  8.7920e-01, -1.5658e+00,  5.8326e-02,
          -4.5617e-02, -1